# Signal Orientation Diagnostic

This notebook checks if signal and sequence are properly aligned by:
1. Checking if move table positions increase monotonically (they should)
2. Extracting chunks around motifs and visualizing signal vs sequence
3. Computing signal-to-base correlations to detect reversal

## Background

**Expected behavior for correctly oriented data:**
- Move table seq-to-sig mapping should be **monotonically INCREASING**
- Base 0 should map to early signal indices (e.g., 0-100)
- Base 1 should map to later signal indices (e.g., 100-200)

**If signal is reversed:**
- Move table mappings would be **monotonically DECREASING**
- Base 0 would map to late signal indices (e.g., 9900-10000)
- Base 1 would map to earlier signal indices (e.g., 9800-9900)

## Key Issue

- **POD5 signal**: Stored in collection order (3' → 5', adaptor enters first)
- **BAM sequence**: Reported in standard convention (5' → 3')
- **Move tables**: Map sequence positions (5' → 3') to signal indices

If the signal orientation doesn't match the sequence orientation, we need to reverse the signal array before analysis.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotnine as p9

from leech.features import extract_move_table
from leech.io.bam_reader import BAMReader
from leech.io.pod5_reader import POD5Reader

# Set plotnine theme
p9.theme_set(p9.theme_minimal())

print("✓ Imports successful")

## Configuration

In [ ]:
# Sample to analyze
SAMPLE = "ala_synthetic"

# Paths (adjust for your environment)
BASE_DIR = Path("/scratch/alpine/jhesselberth@xsede.org/leech/synthetic-trna")
BAM_PATH = BASE_DIR / "bam" / "rebasecall" / SAMPLE / f"{SAMPLE}.aligned.bam"
POD5_PATH = BASE_DIR / "pod5" / SAMPLE / f"{SAMPLE}.pod5"

# Analysis parameters
MOTIF = "CCATGGC"  # CCA-[aa as T]-GGC
MIN_MAPQ = 30  # Minimum mapping quality
N_READS_MONOTONICITY = 500  # Number of reads for monotonicity check
N_READS_VISUALIZATION = 5  # Number of reads for visualization

print("=" * 80)
print("SIGNAL ORIENTATION DIAGNOSTIC CONFIGURATION")
print("=" * 80)
print(f"Sample: {SAMPLE}")
print(f"BAM:    {BAM_PATH}")
print(f"POD5:   {POD5_PATH}")
print(f"Motif:  {MOTIF}")
print(f"MAPQ:   >={MIN_MAPQ}")
print("=" * 80)

## Helper Functions

In [ ]:
def check_move_table_monotonicity(bam_path: Path, n_reads: int = 100, min_mapq: int = 10) -> dict:
    """
    Check if move table mappings are monotonically increasing.

    Returns:
        dict with counts: {"increasing": int, "decreasing": int, "mixed": int, "total": int}
    """
    reader = BAMReader(bam_path, min_mapq=min_mapq)
    results = {"increasing": 0, "decreasing": 0, "mixed": 0, "total": 0}

    with reader:
        for i, aln in enumerate(reader.iter_alignments()):
            if i >= n_reads:
                break
            if aln.is_unmapped or not aln.has_tag("mv"):
                continue

            try:
                move_table = extract_move_table(aln)
                seq_to_sig = move_table.to_seq_to_sig_map()

                # Check if monotonically increasing or decreasing
                diffs = np.diff(seq_to_sig)
                if np.all(diffs >= 0):
                    results["increasing"] += 1
                elif np.all(diffs <= 0):
                    results["decreasing"] += 1
                else:
                    results["mixed"] += 1
                results["total"] += 1

            except Exception as e:
                print(f"  Warning: Failed to process {aln.query_name}: {e}")
                continue

    return results


def extract_motif_chunk_with_signal(
    bam_path: Path,
    pod5_path: Path,
    motif: str = "CCAGGC",
    n_reads: int = 5,
    min_mapq: int = 10,
) -> list[dict]:
    """
    Extract chunks around motifs and their corresponding signal.

    Returns:
        List of dicts with:
        - read_id: read identifier
        - sequence: the motif + context
        - signal: raw signal for that region
        - per_base_means: per-base mean signal
        - seq_to_sig_map: mapping from bases to signal indices
        - motif_start: position of motif start in chunk
    """
    bam_reader = BAMReader(bam_path, min_mapq=min_mapq)
    pod5_reader = POD5Reader(pod5_path)
    chunks = []

    with bam_reader, pod5_reader:
        for aln in bam_reader.iter_alignments():
            if len(chunks) >= n_reads:
                break
            if aln.is_unmapped or not aln.has_tag("mv"):
                continue

            # Find motif in sequence
            seq = aln.query_sequence
            if seq is None:
                continue
            motif_pos = seq.find(motif)
            if motif_pos == -1:
                continue

            try:
                # Get move table and signal
                move_table = extract_move_table(aln)
                seq_to_sig = move_table.to_seq_to_sig_map()

                if aln.query_name is None:
                    continue

                signal, _ = pod5_reader.get_signal(aln.query_name)

                # Extract context around motif
                context = 10
                start = max(0, motif_pos - context)
                end = min(len(seq), motif_pos + len(motif) + context)

                chunk_seq = seq[start:end]
                chunk_sig_start = seq_to_sig[start]
                chunk_sig_end = seq_to_sig[end - 1] if end < len(seq_to_sig) else len(signal)

                chunk_signal = signal[chunk_sig_start:chunk_sig_end]
                chunk_seq_to_sig = seq_to_sig[start:end] - chunk_sig_start

                # Compute per-base mean signal
                per_base_means = []
                for i in range(len(chunk_seq)):
                    sig_start = chunk_seq_to_sig[i]
                    sig_end = (
                        chunk_seq_to_sig[i + 1]
                        if i + 1 < len(chunk_seq_to_sig)
                        else len(chunk_signal)
                    )
                    base_signal = chunk_signal[sig_start:sig_end]
                    per_base_means.append(np.mean(base_signal) if len(base_signal) > 0 else 0)

                chunks.append(
                    {
                        "read_id": aln.query_name,
                        "sequence": chunk_seq,
                        "signal": chunk_signal,
                        "per_base_means": np.array(per_base_means),
                        "seq_to_sig_map": chunk_seq_to_sig,
                        "motif_start": motif_pos - start,
                        "motif_end": motif_pos - start + len(motif),
                    }
                )

            except Exception as e:
                print(f"  Warning: Failed to extract chunk from {aln.query_name}: {e}")
                continue

    return chunks


def plot_signal_alignment(chunks: list[dict]):
    """
    Plot signal traces with sequence annotations using plotnine.

    If signal is reversed, signal features won't match the bases.
    """
    # Prepare data for plotnine
    signal_data = []
    base_data = []

    for chunk in chunks[:5]:  # Limit to 5 reads for visualization
        read_id = chunk["read_id"]

        # Signal trace data
        for idx, sig_val in enumerate(chunk["signal"]):
            signal_data.append(
                {
                    "read_id": read_id,
                    "signal_index": idx,
                    "signal": sig_val,
                }
            )

        # Base annotation data
        for base_idx, (sig_pos, base) in enumerate(
            zip(chunk["seq_to_sig_map"], chunk["sequence"], strict=False)
        ):
            is_motif = base_idx >= chunk["motif_start"] and base_idx < chunk["motif_end"]
            base_data.append(
                {
                    "read_id": read_id,
                    "signal_index": sig_pos,
                    "base": base,
                    "is_motif": is_motif,
                    "y_position": chunk["signal"].max() * 0.95,  # Near top
                }
            )

    signal_df = pd.DataFrame(signal_data)
    base_df = pd.DataFrame(base_data)

    # Create plot
    plot = (
        p9.ggplot()
        # Signal trace
        + p9.geom_line(
            p9.aes(x="signal_index", y="signal"),
            data=signal_df,
            alpha=0.7,
            size=0.3,
            color="#2C3E50",
        )
        # Base boundaries (vertical lines)
        + p9.geom_vline(
            p9.aes(xintercept="signal_index"),
            data=base_df,
            color="gray",
            alpha=0.2,
            size=0.3,
        )
        # Base labels
        + p9.geom_text(
            p9.aes(x="signal_index", y="y_position", label="base", color="is_motif"),
            data=base_df,
            size=7,
            nudge_x=5,
        )
        + p9.scale_color_manual(values={True: "#E74C3C", False: "#34495E"})
        + p9.facet_wrap("~read_id", ncol=1, scales="free")
        + p9.labs(
            title="Signal Alignment Check",
            subtitle="Motif bases in red. If signal is reversed, bases won't align with signal features.",
            x="Signal Index",
            y="Signal (pA)",
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(14, 3 * min(len(chunks), 5)),
            legend_position="none",
            strip_text=p9.element_text(size=9, weight="bold"),
            plot_title=p9.element_text(size=12, weight="bold"),
            plot_subtitle=p9.element_text(size=10),
        )
    )

    plot.show()


def compute_base_signal_correlation(chunks: list[dict]) -> dict:
    """
    Compute mean signal per base type.

    If signal is reversed, correlations may be weaker or inverted.
    """
    base_signals = {"A": [], "C": [], "G": [], "T": [], "U": []}

    for chunk in chunks:
        for base, mean_sig in zip(chunk["sequence"], chunk["per_base_means"], strict=False):
            if base in base_signals:
                base_signals[base].append(mean_sig)

    # Compute mean signal per base
    base_means = {}
    for base, signals in base_signals.items():
        if len(signals) > 0:
            base_means[base] = np.mean(signals)

    return base_means


print("✓ Helper functions defined")

## Test 1: Check Move Table Monotonicity

This checks whether the seq-to-sig mapping is increasing or decreasing.

In [ ]:
print("=" * 80)
print("TEST 1: MOVE TABLE MONOTONICITY")
print("=" * 80)
print(f"Analyzing up to {N_READS_MONOTONICITY} reads with MAPQ >= {MIN_MAPQ}")
print()

monotonicity = check_move_table_monotonicity(
    BAM_PATH, n_reads=N_READS_MONOTONICITY, min_mapq=MIN_MAPQ
)

print(f"\nResults from {monotonicity['total']} reads:")
print(
    f"  Increasing: {monotonicity['increasing']} reads ({monotonicity['increasing']/monotonicity['total']*100:.1f}%)"
)
print(
    f"  Decreasing: {monotonicity['decreasing']} reads ({monotonicity['decreasing']/monotonicity['total']*100:.1f}%)"
)
print(
    f"  Mixed:      {monotonicity['mixed']} reads ({monotonicity['mixed']/monotonicity['total']*100:.1f}%)"
)

print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)
if monotonicity["decreasing"] > 0:
    print("⚠️  FOUND DECREASING MAPPINGS!")
    print("   This suggests signal and sequence are in opposite orientations.")
    print("   Signal may need to be REVERSED before analysis.")
elif monotonicity["increasing"] == monotonicity["total"]:
    print("✓ All mappings are INCREASING (expected for correctly oriented data)")
    print("  Signal and sequence appear to be in the same orientation.")
else:
    print("⚠️  MIXED RESULTS")
    print(f"   {monotonicity['increasing']} increasing, {monotonicity['decreasing']} decreasing")
    print("   This is unexpected - may indicate data quality issues.")
print("=" * 80)

## Test 2: Extract Chunks Around Motif

In [ ]:
print("=" * 80)
print("TEST 2: EXTRACT CHUNKS AROUND MOTIF")
print("=" * 80)
print(f"Searching for motif: {MOTIF}")
print(f"Extracting up to {N_READS_VISUALIZATION} chunks with MAPQ >= {MIN_MAPQ}")
print()

chunks = extract_motif_chunk_with_signal(
    BAM_PATH, POD5_PATH, motif=MOTIF, n_reads=N_READS_VISUALIZATION, min_mapq=MIN_MAPQ
)

print(f"\n✓ Extracted {len(chunks)} chunks")

if len(chunks) > 0:
    print("\nChunk details:")
    for i, chunk in enumerate(chunks):
        print(f"  {i+1}. Read: {chunk['read_id']}")
        print(f"     Sequence length: {len(chunk['sequence'])} bases")
        print(f"     Signal length:   {len(chunk['signal'])} samples")
        print(f"     Motif at:        position {chunk['motif_start']}")
else:
    print("⚠️  No chunks extracted - motif not found in any reads")

print("=" * 80)

## Test 3: Visualize Signal Alignment

Plot signal with base annotations. If signal is reversed, bases will appear "backwards" relative to signal features.

In [ ]:
if len(chunks) > 0:
    print("=" * 80)
    print("TEST 3: VISUALIZE SIGNAL ALIGNMENT")
    print("=" * 80)

    plot_signal_alignment(chunks)

    print("\nLook for:")
    print("  • Do bases align with corresponding signal features?")
    print("  • Do you see clear transitions at base boundaries?")
    print("  • Does the motif (in red) show consistent signal patterns?")
    print("=" * 80)
else:
    print("⚠️  Skipping visualization - no chunks available")

## Test 4: Base-Signal Correlations

In [ ]:
if len(chunks) > 0:
    print("=" * 80)
    print("TEST 4: BASE-SIGNAL CORRELATIONS")
    print("=" * 80)

    base_means = compute_base_signal_correlation(chunks)

    # Create bar plot with plotnine
    base_df = pd.DataFrame(
        [(base, mean) for base, mean in base_means.items()], columns=["base", "mean_signal"]
    )

    plot = (
        p9.ggplot(base_df, p9.aes(x="base", y="mean_signal", fill="base"))
        + p9.geom_col(alpha=0.8)
        + p9.scale_fill_manual(
            values={"A": "#27AE60", "C": "#3498DB", "G": "#E67E22", "T": "#E74C3C", "U": "#9B59B6"}
        )
        + p9.labs(
            title="Mean Signal per Base Type",
            subtitle="Different bases should have distinct signal levels (typical order: G > C > A > U)",
            x="Base",
            y="Mean Signal (pA)",
        )
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(8, 5),
            legend_position="none",
            plot_title=p9.element_text(size=12, weight="bold"),
            plot_subtitle=p9.element_text(size=10),
        )
    )

    plot.show()

    print("\nMean signal per base:")
    for base in sorted(base_means.keys()):
        print(f"  {base}: {base_means[base]:7.2f} pA")

    print("=" * 80)
else:
    print("⚠️  Skipping correlations - no chunks available")

## Summary and Interpretation

### What to look for:

1. **Monotonicity Check**:
   - ✓ All increasing → Signal and sequence are aligned
   - ⚠️ All decreasing → Signal needs to be reversed
   - ⚠️ Mixed → Data quality issues

2. **Visual Alignment**:
   - Bases should align with corresponding signal features
   - Motif should show consistent patterns across reads
   - If bases appear "backwards", signal may be reversed

3. **Base Correlations**:
   - Different bases should have distinct signal levels
   - If correlations are weak, signal orientation may be incorrect

### Key Insight:

- **POD5 signal**: Stored in collection order (3' → 5')
- **BAM sequence**: Reported in standard order (5' → 3')
- **Move tables**: Map sequence to signal

If move tables are decreasing, the signal array needs to be reversed before computing features like dwell times.